In [1]:
import os, sys, re
import pandas as pd

In [2]:
pd.set_option('display.max_colwidth', None)

In [3]:
DATA_FILE = "../results/samples_for_simple_sentences_eltn80_results_analyzed.csv"

In [4]:
n80 = pd.read_csv(DATA_FILE, sep=",", encoding="utf-8")

n80["verb_compound"] = n80["verb_compound"].fillna("")
n80["tag"] = n80["tag"].fillna("")
n80["tags"] = n80["tags"].fillna("")

# ELT n80, mis ei ole ELT
n80_noelt = n80[(n80["tag"]!= "E") & (n80["tag"]!= "L") & (n80["tag"]!= "T")  & ~(n80["tags"].str.contains("ELT", na=False))]
n80_noelt = n80_noelt[(~n80_noelt["verb_head_same_sent"].isna()) & (~n80_noelt["head_verbi_alluv"].isna())]
n80_noelt = n80_noelt[(n80_noelt["tags"]!="") | (n80_noelt["tag"]!="")]

n80_noelt = n80_noelt[['sentence_id', 'head_id', 'head_loc', 'verb', 'verb_compound',
       'morph_case', 'lemma', 'form', 'sentence', 'simple_sentences_gpt4o', 'tags', 'tag',
       'initial_cat', 'verb_head_same_sent', 'head_verbi_alluv']]

# ELT n80, mis on ELT
n80_elt = n80[(n80["tag"]!= "A") & (n80["tag"]!= "S") & ~(n80["tags"].str.contains("AS", na=False))]
#n80_elt = n80[(n80["tag"]== "E") | (n80["tag"]== "L") | (n80["tag"]== "T")  | (n80["tags"].str.contains("ELT", na=False))]
n80_elt = n80_elt[(~n80_elt["verb_head_same_sent"].isna()) & (~n80_elt["head_verbi_alluv"].isna())]
n80_elt = n80_elt[(n80_elt["tags"]!="") | (n80_elt["tag"]!="")]
n80_elt = n80_elt[['sentence_id', 'head_id', 'head_loc', 'verb', 'verb_compound',
       'morph_case', 'lemma', 'form', 'sentence', 'simple_sentences_gpt4o', 'tags', 'tag',
       'initial_cat', 'verb_head_same_sent', 'head_verbi_alluv']]


# Kokkuvõte ELT n80, A ja S (44 näidet)

In [5]:
same_sent = n80_noelt[n80_noelt["verb_head_same_sent"]=="y"]
not_same_sent = n80_noelt[n80_noelt["verb_head_same_sent"]=="n"]
not_same_but_error = not_same_sent[not_same_sent["head_verbi_alluv"]=="y"]
not_same_not_alluv = not_same_sent[not_same_sent["head_verbi_alluv"]=="n"]
not_same_orig_error =  not_same_sent[not_same_sent["head_verbi_alluv"]=="e"]
same_sent_alluv = same_sent[same_sent["head_verbi_alluv"]=="y"]
same_sent_replaced = same_sent[same_sent["head_verbi_alluv"]=="?"]
same_sent_kesksona = same_sent[same_sent["head_verbi_alluv"]=="k"]
same_sent_error = same_sent[same_sent["head_verbi_alluv"]=="e"]
#errors = n80_noelt[n80_noelt["verb_head_same_sent"]=="e"]
gpt_errors = n80_noelt[n80_noelt["verb_head_same_sent"]=="?"]

In [6]:
print("verb + peasõna on samas lihtlauses: ", len(same_sent))
print("verb+ps samas lihtlauses ja ps on verbi alluv: ", len(same_sent_alluv), "/", len(same_sent))
print("verb+ps samas lihtlauses aga peasõna on asendatud teise sõnaga (enamasti sisuliselt õige): ", len(same_sent_replaced), "/", len(same_sent))
print("verb+ps samas lihtlauses aga orig lauses kesksõna vorm: ", len(same_sent_kesksona), "/", len(same_sent))
print("verb+ps samas lihtlauses aga lauses probleem: ", len(same_sent_error), "/", len(same_sent))


print("\n")

print("verb + peasõna pole samas lihtlauses: ", len(not_same_sent))
print("verb+ps pole samas lihtlauses aga orig lauses on ps verbi alluv (gpt lausestamise viga):", len(not_same_but_error), "/", len(not_same_sent) )
print("verb+ps pole samas lihtlauses ja ps ei ole orig lauses verbi alluv:", len(not_same_not_alluv), "/", len(not_same_sent) )
print("verb+ps pole samas lihtlauses ja orig lauses midagi valesti:", len(not_same_orig_error), "/", len(not_same_sent) )

print("\n")

#print("peasõna süntaksi viga:", len(errors))
print("gpt lausestamisega muutus struktuur liiga palju:", len(gpt_errors))


verb + peasõna on samas lihtlauses:  34
verb+ps samas lihtlauses ja ps on verbi alluv:  19 / 34
verb+ps samas lihtlauses aga peasõna on asendatud teise sõnaga (enamasti sisuliselt õige):  0 / 34
verb+ps samas lihtlauses aga orig lauses kesksõna vorm:  0 / 34
verb+ps samas lihtlauses aga lauses probleem:  15 / 34


verb + peasõna pole samas lihtlauses:  10
verb+ps pole samas lihtlauses aga orig lauses on ps verbi alluv (gpt lausestamise viga): 4 / 10
verb+ps pole samas lihtlauses ja ps ei ole orig lauses verbi alluv: 0 / 10
verb+ps pole samas lihtlauses ja orig lauses midagi valesti: 6 / 10


gpt lausestamisega muutus struktuur liiga palju: 0


### Segadusmaatriks

In [7]:
sm_noelt = n80_noelt[n80_noelt["verb_head_same_sent"]!= "?"]
sm_noelt = sm_noelt[(sm_noelt["head_verbi_alluv"]!= "?") & (sm_noelt["head_verbi_alluv"]!= "k")]

In [8]:
pd.crosstab(
    sm_noelt["verb_head_same_sent"],
    sm_noelt["head_verbi_alluv"],
    rownames=["SameSent"],
    colnames=["Alluv"]
)

Alluv,e,y
SameSent,,
n,6,4
y,15,19


### Näited

In [9]:
not_same_sent 

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,simple_sentences_gpt4o,tags,tag,initial_cat,verb_head_same_sent,head_verbi_alluv
3,11434534,18321933,4,nälgima,,in,remondijärk,remondijärgus,Tema nälgis oma remondijärgus luksuskorteris Tallinna südalinnas ( vaatega Estonia teatrile ) .,"['Tema nälgis oma luksuskorteris.', 'Korter oli remondijärgus.', 'Korter asus Tallinna südalinnas.', 'Korterist avanes vaade Estonia teatrile.']",,S,elt_n80_as,n,y
8,6658520,10714082,18,leidma,,abl,kompanii,kompaniilt,"Kalmistu ülevaataja Wayne Bloomquist oli päris üllatunud , kui leidis tema surnuaeda maetud inimese nimele tulnud telefoniarve kompaniilt Sprint .","['Kalmistu ülevaataja on Wayne Bloomquist.', 'Wayne Bloomquist leidis telefoniarve.', 'Telefoniarve oli saadetud tema surnuaeda maetud inimese nimele.', 'Telefoniarve tuli kompaniilt Sprint.', 'Wayne Bloomquist oli päris üllatunud.']",|A|AE|AL|AS|AT|AET|ALT|AST|,,elt_n80_as,n,y
9,15078520,23604634,7,helistama,,el,naine,naisest,Siis on veel mingi lõik vanast naisest kes helistab politseisse ja kurdab vanilla ninja kontserdi üle.,"['Vanast naisest on mingi lõik.', 'Vana naine helistab politseisse.', 'Vana naine kurdab Vanilla Ninja kontserdi üle.']",|A|AE|AL|AS|AT|AET|ALT|AST|,A,elt_n80_as,n,e
25,2283231,3655369,11,kommenteerima,,in,võidujoovastus,võidujoovastuses,""" See on küll kõige väiksem mure , "" kommenteeris võidujoovastuses EOK-i president Mart Siimann Veerpalule ja Jaak Maele makstavaid preemiasummasid .","['EOK-i president on Mart Siimann.', 'Mart Siimann kommenteeris Veerpalule ja Jaak Maele makstavaid preemiasummasid.', 'Mart Siimann ütles, et see on kõige väiksem mure.']",,S,elt_n80_as,n,y
38,10729192,17192087,5,taastuma,,in,käsi,käes,"Kui operatsioon õnnestub ja käes verevarustus taastub , pole arstidel ja patsiendil veel võit käes .","['Operatsioon võib õnnestuda.', 'Käes võib verevarustus taastuda.', 'Arstidel ja patsiendil pole veel võit käes.']",,S,elt_n80_as,n,e
45,13502145,21595462,9,sõitma,maha,in,joove,joobes,Tund pärast südaööd Karja tänaval kihutades sõitis raskes joobes kohtlajärvelane Centrali restorani ees maha laternaposti ja põrkas vastu tuletõrjemaja seina .,"['Tund pärast südaööd kihutas Karja tänaval raskes joobes kohtlajärvelane.', 'Ta sõitis Centrali restorani ees maha laternaposti.', 'Ta põrkas vastu tuletõrjemaja seina.']",|S|AS|LS|ST|AST|LST|,S,elt_n80_as,n,e
53,14076208,22413034,1,lähenema,,ad,vanahärra,Vanahärral,"Vanahärral ta iga lähenes juba 80-le oli tavaks lugeda , kirjutada või malet mängida kella 3-4 hommikul ning magada siis kella 11.","['Vanahärral oli tavaks lugeda, kirjutada või malet mängida kella 3-4 hommikul.', 'Vanahärra iga lähenes juba 80-le.', 'Vanahärral oli tavaks magada siis kella 11-ni.']",|A|AE|AL|AS|AT|AET|ALT|AST|,A,elt_n80_as,n,e
84,4057693,6531215,10,võtma,ette,in,tingimus,tingimustes,"Kaitseväe ellujäämiskursusi viiakse läbi regulaarselt , kuid nii rasketes tingimustes veetakistuse läbimist nagu rahuvalvajad üleeile Pakri saarelt ette võtsid , pole varem Eestis tehtud .","['Kaitseväe ellujäämiskursusi viiakse läbi regulaarselt.', 'Rahuvalvajad üleeile Pakri saarelt ette võtsid veetakistuse läbimise.', 'See veetakistuse läbimine toimus väga rasketes tingimustes.', 'Sellist veetakistuse läbimist pole varem Eestis tehtud.']",,S,elt_n80_as,n,y
89,5823821,9344735,25,kaasnema,,in,vorm,vormides,"George Lucasi kosmosemütoloogia viimane osa on toonud Londoni , Madridi , Malaisia ja Prantsusmaa tänavaile kostümeeritud fännid , lisaks ilmestavad linnaruumi reklaamikampaaniaga kaasnevad valgetes vormides vaenuvägede rivid .","['George Lucasi kosmosemütoloogia viimane osa on toonud tänavaile kostümeeritud fännid.', 'Kostümeeritud fännid on Londoni tänavail.', 'Kostümeeritud fännid on Madridi tänavail.', 'Kostümeeritud fännid on Malaisia tänavail.', 'Kostümeeritud fännid on Prantsusmaa tänavail.', 'Linnaruumi ilmestavad reklaamikampaaniaga kaasnevad vaenuvägede rivid.', 'Vaenuvägede rivid on valgetes vo

In [10]:
not_same_but_error

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,simple_sentences_gpt4o,tags,tag,initial_cat,verb_head_same_sent,head_verbi_alluv
3,11434534,18321933,4,nälgima,,in,remondijärk,remondijärgus,Tema nälgis oma remondijärgus luksuskorteris Tallinna südalinnas ( vaatega Estonia teatrile ) .,"['Tema nälgis oma luksuskorteris.', 'Korter oli remondijärgus.', 'Korter asus Tallinna südalinnas.', 'Korterist avanes vaade Estonia teatrile.']",,S,elt_n80_as,n,y
8,6658520,10714082,18,leidma,,abl,kompanii,kompaniilt,"Kalmistu ülevaataja Wayne Bloomquist oli päris üllatunud , kui leidis tema surnuaeda maetud inimese nimele tulnud telefoniarve kompaniilt Sprint .","['Kalmistu ülevaataja on Wayne Bloomquist.', 'Wayne Bloomquist leidis telefoniarve.', 'Telefoniarve oli saadetud tema surnuaeda maetud inimese nimele.', 'Telefoniarve tuli kompaniilt Sprint.', 'Wayne Bloomquist oli päris üllatunud.']",|A|AE|AL|AS|AT|AET|ALT|AST|,,elt_n80_as,n,y
25,2283231,3655369,11,kommenteerima,,in,võidujoovastus,võidujoovastuses,""" See on küll kõige väiksem mure , "" kommenteeris võidujoovastuses EOK-i president Mart Siimann Veerpalule ja Jaak Maele makstavaid preemiasummasid .","['EOK-i president on Mart Siimann.', 'Mart Siimann kommenteeris Veerpalule ja Jaak Maele makstavaid preemiasummasid.', 'Mart Siimann ütles, et see on kõige väiksem mure.']",,S,elt_n80_as,n,y
84,4057693,6531215,10,võtma,ette,in,tingimus,tingimustes,"Kaitseväe ellujäämiskursusi viiakse läbi regulaarselt , kuid nii rasketes tingimustes veetakistuse läbimist nagu rahuvalvajad üleeile Pakri saarelt ette võtsid , pole varem Eestis tehtud .","['Kaitseväe ellujäämiskursusi viiakse läbi regulaarselt.', 'Rahuvalvajad üleeile Pakri saarelt ette võtsid veetakistuse läbimise.', 'See veetakistuse läbimine toimus väga rasketes tingimustes.', 'Sellist veetakistuse läbimist pole varem Eestis tehtud.']",,S,elt_n80_as,n,y


In [11]:
not_same_not_alluv

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,simple_sentences_gpt4o,tags,tag,initial_cat,verb_head_same_sent,head_verbi_alluv


In [12]:
not_same_orig_error

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,simple_sentences_gpt4o,tags,tag,initial_cat,verb_head_same_sent,head_verbi_alluv
9,15078520,23604634,7,helistama,,el,naine,naisest,Siis on veel mingi lõik vanast naisest kes helistab politseisse ja kurdab vanilla ninja kontserdi üle.,"['Vanast naisest on mingi lõik.', 'Vana naine helistab politseisse.', 'Vana naine kurdab Vanilla Ninja kontserdi üle.']",|A|AE|AL|AS|AT|AET|ALT|AST|,A,elt_n80_as,n,e
38,10729192,17192087,5,taastuma,,in,käsi,käes,"Kui operatsioon õnnestub ja käes verevarustus taastub , pole arstidel ja patsiendil veel võit käes .","['Operatsioon võib õnnestuda.', 'Käes võib verevarustus taastuda.', 'Arstidel ja patsiendil pole veel võit käes.']",,S,elt_n80_as,n,e
45,13502145,21595462,9,sõitma,maha,in,joove,joobes,Tund pärast südaööd Karja tänaval kihutades sõitis raskes joobes kohtlajärvelane Centrali restorani ees maha laternaposti ja põrkas vastu tuletõrjemaja seina .,"['Tund pärast südaööd kihutas Karja tänaval raskes joobes kohtlajärvelane.', 'Ta sõitis Centrali restorani ees maha laternaposti.', 'Ta põrkas vastu tuletõrjemaja seina.']",|S|AS|LS|ST|AST|LST|,S,elt_n80_as,n,e
53,14076208,22413034,1,lähenema,,ad,vanahärra,Vanahärral,"Vanahärral ta iga lähenes juba 80-le oli tavaks lugeda , kirjutada või malet mängida kella 3-4 hommikul ning magada siis kella 11.","['Vanahärral oli tavaks lugeda, kirjutada või malet mängida kella 3-4 hommikul.', 'Vanahärra iga lähenes juba 80-le.', 'Vanahärral oli tavaks magada siis kella 11-ni.']",|A|AE|AL|AS|AT|AET|ALT|AST|,A,elt_n80_as,n,e
89,5823821,9344735,25,kaasnema,,in,vorm,vormides,"George Lucasi kosmosemütoloogia viimane osa on toonud Londoni , Madridi , Malaisia ja Prantsusmaa tänavaile kostümeeritud fännid , lisaks ilmestavad linnaruumi reklaamikampaaniaga kaasnevad valgetes vormides vaenuvägede rivid .","['George Lucasi kosmosemütoloogia viimane osa on toonud tänavaile kostümeeritud fännid.', 'Kostümeeritud fännid on Londoni tänavail.', 'Kostümeeritud fännid on Madridi tänavail.', 'Kostümeeritud fännid on Malaisia tänavail.', 'Kostümeeritud fännid on Prantsusmaa tänavail.', 'Linnaruumi ilmestavad reklaamikampaaniaga kaasnevad vaenuvägede rivid.', 'Vaenuvägede rivid on valgetes vormides.']",,S,elt_n80_as,n,e
95,17764111,27121867,32,pidama,,ill,piin,piinadesse,"meil oli suur punane kass aga ta j2i nii haigeks , et ma pidin tiina toometi kliinikus kiisukese 400 eegueest magamaa panema ... see oli kurb aga kui ta oleks pidanud piinadesse lõpuks koolema see oleks veel jubedam olnud aga miski ravimine oleks jube kalliks läinud ja noh .. tegelikult oli see ema kass","['Meil oli suur punane kass.', 'Kass jäi väga haigeks.', 'Ma pidin viima kassi Tiina Toometi kliinikusse.', 'Ma pidin panema kassi magama.', 'See maksis 400 krooni.', 'See oli kurb.', 'Kui kass oleks piinadesse lõpuks surnud, oleks see olnud veel jubedam.', 'Ravimine oleks olnud väga kallis.', 'Tegelikult oli see ema kass.']",,S,elt_n80_as,n,e


In [13]:
same_sent.sample(5)

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,simple_sentences_gpt4o,tags,tag,initial_cat,verb_head_same_sent,head_verbi_alluv
1,8382028,13431809,12,jaguma,,in,sobivus,sobivuses,"Mäletan , et filmi valmimisjärgus jagus palju kahtlejaid Julia Ormondi nimiossa sobivuses .","['Mäletan filmi valmimisjärku.', 'Filmi valmimisjärgus jagus palju kahtlejaid.', 'Kahtlejad kahtlesid Julia Ormondi nimiossa sobivuses.']",,S,elt_n80_as,y,e
85,14581191,23030035,2,paistma,,in,erutusseisund,erutusseisundis,"Niiet erutusseisundis paistad sina olevat 4. "" läheme ... algküsimuse juurde tagasi ... "" Palun .","['Erutusseisundis paistad sina olevat.', 'Läheme algküsimuse juurde tagasi.', 'Palun.']",|S|AS|LS|ST|AST|LST|,S,elt_n80_as,y,y
60,21064516,29900904,4,soovima,,in,Ade,Ades,+Ades: Ades soovib kõigile siin viibivatele seksidele ... mulatidele .... lullidele .... neiudele ..... ja kesekesekesekesele hääh öööd,"['Ades soovib kõigile siin viibivatele seksidele head ööd.', 'Ades soovib kõigile siin viibivatele mulatidele head ööd.', 'Ades soovib kõigile siin viibivatele lullidele head ööd.', 'Ades soovib kõigile siin viibivatele neiudele head ööd.', 'Ades soovib kõigile siin viibivatele kesekesekesekesele head ööd.']",,A,elt_n80_as,y,e
56,11931,20494,5,esinema,,ad,ajakirjanik,ajakirjanikul,Sellist vedamist esineb ühel ajakirjanikul harva .,['Sellist vedamist esineb ühel ajakirjanikul harva.'],|A|AE|AL|AS|AT|AET|ALT|AST|,A,elt_n80_as,y,y
67,3461780,5574216,10,jooksma,,all,miljonär,miljonärile,"TV3 kanalil jookseb lisaks maailmas ülimenukaks osutunud "" Vabale miljonärile "" veel "" Õige valik "" ning kordusena "" Farm "" , mille omakorda asendab peatselt draamaseriaal "" Kodu keset linna "" .","['TV3 kanalil jookseb maailmas ülimenukaks osutunud ""Vaba miljonär"".', 'TV3 kanalil jookseb ""Õige valik"".', 'TV3 kanalil jookseb kordusena ""Farm"".', '""Farm"" asendab peatselt draamaseriaal ""Kodu keset linna"".']",|A|AE|AL|AS|AT|AET|ALT|AST|,,elt_n80_as,y,e


In [14]:
same_sent_alluv.sample(10)

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,simple_sentences_gpt4o,tags,tag,initial_cat,verb_head_same_sent,head_verbi_alluv
56,11931,20494,5,esinema,,ad,ajakirjanik,ajakirjanikul,Sellist vedamist esineb ühel ajakirjanikul harva .,['Sellist vedamist esineb ühel ajakirjanikul harva.'],|A|AE|AL|AS|AT|AET|ALT|AST|,A,elt_n80_as,y,y
76,10227305,16411917,17,tegutsema,,el,kolleeg,kolleegidest,"Eesti sõjaväemeedikuid abistas vigastatute turgutamisel ka USA merejalaväe meditsiinitöötaja , kes Kuke hinnangul tegutses oma Eesti kolleegidest ebaprofessionaalsemalt .","['Eesti sõjaväemeedikuid abistas vigastatute turgutamisel USA merejalaväe meditsiinitöötaja.', 'Kuke hinnangul tegutses USA merejalaväe meditsiinitöötaja oma Eesti kolleegidest ebaprofessionaalsemalt.']",|A|AE|AL|AS|AT|AET|ALT|AST|,A,elt_n80_as,y,y
26,7276985,11696920,5,põhjustama,,ad,katselehm,katselehmadel,"Kui K2CO3 lisasöötmine põhjustas katselehmadel kuivaine söömuse vähenemist , siis karbamiidi lisasöötmine ei mõjutanud statistiliselt oluliselt ei kuivaine ega teiste uuritud toitefaktorite söömust .","['K2CO3 lisasöötmine põhjustas katselehmadel kuivaine söömuse vähenemist.', 'Karbamiidi lisasöötmine ei mõjutanud statistiliselt oluliselt kuivaine söömust.', 'Karbamiidi lisasöötmine ei mõjutanud statistiliselt oluliselt teiste uuritud toitefaktorite söömust.']",,A,elt_n80_as,y,y
77,13782737,21975045,12,tegutsema,,el,inimene,inimestest,"Sellistes olukordades tegutsevad nartsissistid keskmiselt 20percent ; paremini või osavamalt teistest inimestest , kirjutas päevaleht The Independent viitega Blackpoolis räägitule .","['Sellistes olukordades tegutsevad nartsissistid keskmiselt 20% paremini teistest inimestest.', 'Sellistes olukordades tegutsevad nartsissistid keskmiselt 20% osavamalt teistest inimestest.', 'Seda kirjutas päevaleht The Independent.', 'The Independent viitas Blackpoolis räägitule.']",|A|AE|AL|AS|AT|AET|ALT|AST|,A,elt_n80_as,y,y
43,7335710,11769014,3,esinema,,ad,Müller,Mülleril,esinevad juba Mülleril ja on ilmselt tõlkelaenud saksa keelest ( Habicht 2001 : 63-69 ; vt. ka Hasselblatt 1990 ) .,"['Need esinevad juba Mülleril.', 'Need on ilmselt tõlkelaenud saksa keelest.', 'Seda on mainitud Habichti töös (2001: 63-69).', 'Seda on mainitud ka Hasselblatti töös (1990).']",,A,elt_n80_as,y,y
18,8784986,14100284,14,sarnanema,,in,rassiviht,rassivihas,"Ent Venemaa ei sarnane kujunenud olukorras NATOga , pigem sarnaneb ta oma shovinistlikus rassivihas hoopis Serbiaga , kes rakendas genotsiidi separatistlikult meelestatud teiseusuliste suhtes .","['Venemaa ei sarnane NATOga.', 'Venemaa sarnaneb Serbiaga.', 'Venemaa sarnaneb Serbiaga oma shovinistlikus rassivihas.', 'Serbia rakendas genotsiidi separatistlikult meelestatud teiseusuliste suhtes.']",,S,elt_n80_as,y,y
52,14184510,22549174,7,põhjustama,,ad,sina,sul,"Lang : Seesama üks pitsa põhjustab sul suure pildi lehes no parim reklaam , mis üldse olla võib , täiesti tasuta tuleb kätte !","['Lang ütles midagi.', 'Üks pitsa põhjustab sul suure pildi lehes.', 'See on parim reklaam, mis üldse olla võib.', 'See reklaam tuleb täiesti tasuta kätte.']",,A,elt_n80_as,y,y
80,15253064,23824147,6,sõitma,,el,Voldemar,Voldemarist,Vast poolteist kilomeetrit tagasi sõitsime Voldemarist ( 44 ) mööda .,"['Poolteist kilomeetrit tagasi sõitsime Voldemarist mööda.', 'Voldemar on 44-aastane.']",|A|AE|AL|AS|AT|AET|ALT|AST|,A,elt_n80_as,y,y
10,8557729,13716125,19,maksma,kinni,in,tingimus,tingimustes,"Soojus on fikseeritud hinnaga kaup ja seda võib vanas vaimus edasi jagada üksnes juhul , kui keegi tootja tingimustes paratamatu kahjumi talle kinni maksab .","['Soojus on fikseeritud hinnaga kaup.', 'Soojust võib vanas vaimus edasi jagada ainult juhul, kui keegi maksab tootja tingimustes paratamatu kahjumi kinni.']",,S,elt_n80_as,y,y
65,3795925,6113944,19,muutma,,in,käsi,kätes,"Mobiiltelefoni varguse või kadumise puhul saab kasutada mitmeid koode , mis muudavad telefoni või selles oleva SIM-kaardi võõrastes kätes

In [15]:
same_sent_replaced 

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,simple_sentences_gpt4o,tags,tag,initial_cat,verb_head_same_sent,head_verbi_alluv


In [16]:
same_sent_kesksona 

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,simple_sentences_gpt4o,tags,tag,initial_cat,verb_head_same_sent,head_verbi_alluv


In [17]:
same_sent_error 

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,simple_sentences_gpt4o,tags,tag,initial_cat,verb_head_same_sent,head_verbi_alluv
0,14196515,22569505,4,nentima,,in,tagamesi,tagamees,"New York Knicksi tagamees Jamal Crawford nentis ajalehele New York Post , et gay-korvpallur oleks omapärane .","['New York Knicksi tagamees on Jamal Crawford.', 'Jamal Crawford nentis ajalehele New York Post, et gay-korvpallur oleks omapärane.']",,A,elt_n80_as,y,e
1,8382028,13431809,12,jaguma,,in,sobivus,sobivuses,"Mäletan , et filmi valmimisjärgus jagus palju kahtlejaid Julia Ormondi nimiossa sobivuses .","['Mäletan filmi valmimisjärku.', 'Filmi valmimisjärgus jagus palju kahtlejaid.', 'Kahtlejad kahtlesid Julia Ormondi nimiossa sobivuses.']",,S,elt_n80_as,y,e
7,15983761,24854353,1,hakkama,,el,Netskaabakas,Netskaabakast,"Netskaabakast ei hakka ma isegi rääkima , IMHO on too viimane saast .","['Ma ei hakka Netskaabakast rääkima.', 'IMHO järgi on Netskaabakas viimane saast.']",,A,elt_n80_as,y,e
22,3311971,5325276,10,muutma,,in,asu,asus-,"-viia läbi avaliku teenistuse reform , mis muudab asus- tuste juhtimise paindlikumaks ja kaotab senised palgaastmed ( takistuseks võib kujuneda "" ametnike partei "" vastuseis )","['Avaliku teenistuse reform tuleb läbi viia.', 'Reform muudab asutuste juhtimise paindlikumaks.', 'Reform kaotab senised palgaastmed.', 'Takistuseks võib kujuneda ""ametnike partei"" vastuseis.']",|S|AS|LS|ST|AST|LST|,,elt_n80_as,y,e
29,18440013,27943087,41,püüdma,kinni,in,karjavara,karjavaras,"Vist parim näide sellisest ametnikust on 19. saj ameerika sherif - kui kari kadunuks jäi , tulid talunikud kokku , patsutasid mõnele omade seast õlale ja ütlesid : « Sina , mees oskad siin meist paremini lasta , mine püüa karjavaras kinni , meie aitame sind Sina oled meie sherif . »","['19. sajandil oli Ameerikas sherif.', 'Kui kari kadunuks jäi, tulid talunikud kokku.', 'Talunikud patsutasid mõnele omade seast õlale.', 'Talunikud ütlesid: ""Sina, mees, oskad siin meist paremini lasta.""', 'Talunikud ütlesid: ""Mine püüa karjavaras kinni.""', 'Talunikud ütlesid: ""Meie aitame sind.""', 'Talunikud ütlesid: ""Sina oled meie sherif.""']",,A,elt_n80_as,y,e
36,9181110,14756931,7,kärpima,,ad,vajadus,vajadusel,"Tema hinnangul on mõistlik , kui vajadusel kärbib eelarvet valitsus .","['Tema hinnangul on mõistlik, kui vajadusel kärbib eelarvet valitsus.']",,S,elt_n80_as,y,e
44,160863,267637,6,töötama,,el,poja,pojast,"Leili ja Kalju kolmest täiskasvanud pojast töötavad kaks kodutalu metsades , kolmas teenib metsaveotraktoriga raha - mõni kuu 7000-8000 krooni , teine kuu peab Leili oma väiksest sissetulekust teda toetama .","['Leili ja Kalju kolmest täiskasvanud pojast töötavad kaks kodutalu metsades.', 'Kolmas poeg teenib metsaveotraktoriga raha.', 'Mõni kuu teenib ta 7000-8000 krooni.', 'Teine kuu peab Leili oma väiksest sissetulekust teda toetama.']",|A|AE|AL|AS|AT|AET|ALT|AST|,A,elt_n80_as,y,e
60,21064516,29900904,4,soovima,,in,Ade,Ades,+Ades: Ades soovib kõigile siin viibivatele seksidele ... mulatidele .... lullidele .... neiudele ..... ja kesekesekesekesele hääh öööd,"['Ades soovib kõigile siin viibivatele seksidele head ööd.', 'Ades soovib kõigile siin viibivatele mulatidele head ööd.', 'Ades soovib kõigile siin viibivatele lullidele head ööd.', 'Ades soovib kõigile siin viibivatele neiudele head ööd.', 'Ades soovib kõigile siin viibivatele kesekesekesekesele head ööd.']",,A,elt_n80_as,y,e
62,13632228,21799637,10,teadma,,in,Peetri,Peetris,""" Koerad on üldjuhul kellegi omad , "" teadis Peetris .","['Koerad on üldjuhul kellegi omad.', 'Peetris teadis seda.']",|AL|ALT|,A,elt_n80_as,y,e
66,11014037,17641027,8,häirima,,in,salastatud,salastatus,Mind häiris ka see komisjoni töö hirmus salastatus .,['Mind häiris komisjoni töö hirmus salastatus.'],,S,elt_n80_as,y,e


In [16]:
#errors

In [18]:
gpt_errors

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,simple_sentences_gpt4o,tags,tag,initial_cat,verb_head_same_sent,head_verbi_alluv


# Kokkuvõte ELT n80, mis on sõnastiku/gpt poolt saanud E/L/T (83 näidet)

In [19]:
same_sent = n80_elt[n80_elt["verb_head_same_sent"]=="y"]
not_same_sent = n80_elt[n80_elt["verb_head_same_sent"]=="n"]
not_same_but_error = not_same_sent[not_same_sent["head_verbi_alluv"]=="?"]
not_same_not_alluv = not_same_sent[not_same_sent["head_verbi_alluv"]=="n"]
not_same_orig_error =  not_same_sent[not_same_sent["head_verbi_alluv"]=="e"]
same_sent_alluv = same_sent[same_sent["head_verbi_alluv"]=="y"]
same_sent_replaced = same_sent[same_sent["head_verbi_alluv"]=="?"]
same_sent_kesksona = same_sent[same_sent["head_verbi_alluv"]=="k"]
same_sent_error = same_sent[same_sent["head_verbi_alluv"]=="e"]
#errors = n80_elt[n80_elt["verb_head_same_sent"]=="e"]
gpt_errors = n80_elt[n80_elt["verb_head_same_sent"]=="?"]

In [20]:
print("verb + peasõna on samas lihtlauses: ", len(same_sent))
print("verb+ps samas lihtlauses ja ps on verbi alluv: ", len(same_sent_alluv), "/", len(same_sent))
print("verb+ps samas lihtlauses aga peasõna on asendatud teise sõnaga (enamasti sisuliselt õige): ", len(same_sent_replaced), "/", len(same_sent))
print("verb+ps samas lihtlauses aga orig lauses kesksõna vorm: ", len(same_sent_kesksona), "/", len(same_sent))
print("verb+ps samas lihtlauses aga lauses probleem: ", len(same_sent_error), "/", len(same_sent))


print("\n")

print("verb + peasõna pole samas lihtlauses: ", len(not_same_sent))
print("verb+ps pole samas lihtlauses aga orig lauses on ps verbi alluv (gpt lausestamise viga):", len(not_same_but_error), "/", len(not_same_sent) )
print("verb+ps pole samas lihtlauses ja ps ei ole orig lauses verbi alluv:", len(not_same_not_alluv), "/", len(not_same_sent) )
print("verb+ps pole samas lihtlauses ja orig lauses midagi valesti:", len(not_same_orig_error), "/", len(not_same_sent) )

print("\n")

#print("peasõna süntaksi viga:", len(errors))
print("gpt lausestamisega muutus struktuur liiga palju:", len(gpt_errors))


verb + peasõna on samas lihtlauses:  76
verb+ps samas lihtlauses ja ps on verbi alluv:  72 / 76
verb+ps samas lihtlauses aga peasõna on asendatud teise sõnaga (enamasti sisuliselt õige):  1 / 76
verb+ps samas lihtlauses aga orig lauses kesksõna vorm:  0 / 76
verb+ps samas lihtlauses aga lauses probleem:  3 / 76


verb + peasõna pole samas lihtlauses:  6
verb+ps pole samas lihtlauses aga orig lauses on ps verbi alluv (gpt lausestamise viga): 0 / 6
verb+ps pole samas lihtlauses ja ps ei ole orig lauses verbi alluv: 0 / 6
verb+ps pole samas lihtlauses ja orig lauses midagi valesti: 0 / 6


gpt lausestamisega muutus struktuur liiga palju: 1


### Segadusmaatriks

In [21]:
sm_elt = n80_elt[n80_elt["verb_head_same_sent"]!= "?"]
sm_elt.loc[
    (sm_elt["verb_head_same_sent"] == "n") &
    (sm_elt["head_verbi_alluv"] == "?"),
    "head_verbi_alluv"
] = "y"
sm_elt = sm_elt[(sm_elt["head_verbi_alluv"]!= "?")]

In [22]:
pd.crosstab(
    sm_elt["verb_head_same_sent"],
    sm_elt["head_verbi_alluv"],
    rownames=["SameSent"],
    colnames=["Alluv"]
)

Alluv,e,y
SameSent,,
n,0,6
y,3,72


### Näited

In [23]:
not_same_sent 

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,simple_sentences_gpt4o,tags,tag,initial_cat,verb_head_same_sent,head_verbi_alluv
107,10485621,16813554,6,jääma,ilma,in,esilekutsumine,esilekutsumises,Väidetavalt on hiljuti erakorraliste valimiste esilekutsumises valitsuse toetusest ilma jäänud Siimann otsustanud ühe hoobiga nõrgendada oma kabinetis ekspeaminister Tiit Vähi mõju ning vabaneda ebapopulaarsetest ministritest .,"['Siimann on hiljuti kaotanud valitsuse toetuse.', 'Siimann on otsustanud nõrgendada Tiit Vähi mõju oma kabinetis.', 'Tiit Vähi on ekspeaminister.', 'Siimann on otsustanud vabaneda ebapopulaarsetest ministritest.']",,L,elt_n80_elt,n,y
115,1385080,2202753,6,põhjustama,,in,kivi-söeshaht,kivi-söeshahtis,"Ekspertide esialgsetel hinnangutel põhjustas Barakova kivi-söeshahtis 664 meetri sügavusel toimunud plahvatuse rikkis gaasiandur , mis ei näidanud , et shahti on kogunenud suur hulk metaani ja kivisöetolmu üliohtlikku segu .","['Ekspertide esialgsetel hinnangutel põhjustas plahvatuse rikkis gaasiandur.', 'Plahvatus toimus Barakova kivisöeshahtis.', 'Plahvatus toimus 664 meetri sügavusel.', 'Gaasiandur ei näidanud, et shahti on kogunenud suur hulk metaani.', 'Shahti oli kogunenud suur hulk kivisöetolmu üliohtlikku segu.']",,L,elt_n80_elt,n,y
132,13510140,21608429,7,tundma,,in,ametikool,ametikoolis,Samal ajal tunnevad tööandjad suurt puudust ametikoolis hea väljaõppe ja korraliku tööharjumuse saanud spetsialistidest .,"['Samal ajal tunnevad tööandjad suurt puudust spetsialistidest.', 'Spetsialistid on saanud ametikoolis hea väljaõppe.', 'Spetsialistidel on korralik tööharjumus.']",|L|AL|EL|LS|LT|ALT|ELT|LST|,L,elt_n80_elt,n,y
137,2840201,4555037,9,sõnama,,in,Taškent,Taškendis,"Venemaa kaitseminister Sergei Ivanov sõnas kolmapäeval Usbekistani pealinnas Taškendis ajakirjanikele , et kui USA kavatseb jätkab oma raketikilbi rajamist Kesk-Euroopas , siis võib Venemaa paigutada oma raketiüksused Kaliningradi , vahendab BBC .","['Venemaa kaitseminister Sergei Ivanov sõnas kolmapäeval midagi ajakirjanikele.', 'Ta rääkis Usbekistani pealinnas Taškendis.', 'USA kavatseb jätkata oma raketikilbi rajamist Kesk-Euroopas.', 'Venemaa võib paigutada oma raketiüksused Kaliningradi.', 'BBC vahendab seda.']",|L|AL|EL|LS|LT|ALT|ELT|LST|,L,elt_n80_elt,n,y
154,8945593,14379441,9,olema,koht,ad,filmipidu,filmipeol,"Burgsviki festival pole suur , kuid sel iga-aastasel filmipeol on alati kohal üle poole tosina filmilooja kõigist kolmest Balti riigist ja veelgi arvukamalt Rootsi poolelt .","['Burgsviki festival pole suur.', 'See on iga-aastane filmipidu.', 'Sellel festivalil on alati kohal üle poole tosina filmilooja kõigist kolmest Balti riigist.', 'Sellel festivalil on alati kohal veelgi arvukamalt Rootsi poolelt.']",,E,elt_n80_elt,n,y
164,10469540,16789523,10,kordama,,in,Viljandi,Viljandis,"Tänavu on haapsallanna korra juba isiklikku tippmarki korranud - Viljandis , noorte lahtistel meistrivõistlustel alistus 1 .","['Tänavu on haapsallanna korra juba isiklikku tippmarki korranud.', 'See juhtus Viljandis.', 'See juhtus noorte lahtistel meistrivõistlustel.', 'Seal alistus 1.']",|L|AL|EL|LS|LT|ALT|ELT|LST|,E,elt_n80_elt,n,y


In [24]:
not_same_but_error

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,simple_sentences_gpt4o,tags,tag,initial_cat,verb_head_same_sent,head_verbi_alluv


In [25]:
not_same_not_alluv

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,simple_sentences_gpt4o,tags,tag,initial_cat,verb_head_same_sent,head_verbi_alluv


In [26]:
not_same_orig_error

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,simple_sentences_gpt4o,tags,tag,initial_cat,verb_head_same_sent,head_verbi_alluv


In [27]:
same_sent.sample(5)

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,simple_sentences_gpt4o,tags,tag,initial_cat,verb_head_same_sent,head_verbi_alluv
144,5671227,9096096,1,võitlema,,in,Varbola,Varbolas,Varbolas võitleb ta Thori rollis .,['Varbolas võitleb ta Thori rollis.'],|L|AL|EL|LS|LT|ALT|ELT|LST|,L,elt_n80_elt,y,y
131,13517648,21620049,5,lahkuma,,abl,imedemaa,Imedemaalt,""" Igaühele , kes Imedemaalt lahkub , peaks mõni oma tehtud asi kätte jääma , ” ütles Matvei .","['Igaühele, kes Imedemaalt lahkub, peaks mõni oma tehtud asi kätte jääma.', 'Matvei ütles seda.']",|L|AL|EL|LS|LT|ALT|ELT|LST|,L,elt_n80_elt,y,y
135,17486694,26807930,15,paluma,,el,Cibus,Cibusest,"VÕRKPALL : Euroopa Meistrite Liigaks valmistuv ESS Pärnu meeskond palus Tartu Pere Leib/Cibusest appi temporündaja Marek Pihlaku , ent eile Itaaliasse lennanud rivistuses teda siiski polnud .","['ESS Pärnu meeskond valmistub Euroopa Meistrite Liigaks.', 'Meeskond palus Tartu Pere Leib/Cibusest appi temporündaja Marek Pihlaku.', 'Eile lendas rivistus Itaaliasse.', 'Rivistuses Marek Pihlaku siiski polnud.']",|L|AL|EL|LS|LT|ALT|ELT|LST|,L,elt_n80_elt,y,y
166,18858835,28480647,5,andma,järele,ad,turg,turgudel,"Kartulihinna talvine kõrgtase on turgudel veidi järele andnud , odavam import on surunud hinda alla .","['Kartulihinna talvine kõrgtase on turgudel veidi järele andnud.', 'Odavam import on surunud hinda alla.']",,L,elt_n80_elt,y,y
190,13165850,21064784,11,jätkuma,,ill,keskus,keskustesse,"Paisuots ütles , et kuna puudub väljaõppestruktuur , ei jätku keskustesse mehi .","['Paisuots ütles, et puudub väljaõppestruktuur.', 'Paisuots ütles, et keskustesse ei jätku mehi.']",,L,elt_n80_elt,y,y


In [28]:
same_sent_alluv.sample(10)

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,simple_sentences_gpt4o,tags,tag,initial_cat,verb_head_same_sent,head_verbi_alluv
192,18933188,28586001,4,häirima,,in,raadiokanal,raadiokanalites,Mis teid meie raadiokanalites häirib ?,['Mis teid meie raadiokanalites häirib?'],,L,elt_n80_elt,y,y
128,4707999,7564863,2,tuletama,,in,Päevaleht,Päevalehes,"Tänases Päevalehes tuletab Isamaaliidu mees Ammas meelde , et koalitsioonilepingus on ka majandusministeeriumi ja teede- ja sideministeeriumi liitmise punkt .","['Tänases Päevalehes tuletab Isamaaliidu mees Ammas meelde ühte punkti.', 'Koalitsioonilepingus on majandusministeeriumi ja teede- ja sideministeeriumi liitmise punkt.']",|L|AL|EL|LS|LT|ALT|ELT|LST|,L,elt_n80_elt,y,y
193,8098061,12994472,5,tegema,,adit,elulookirjeldus,elulookirjeldusse,"Tavaliselt ei tee naised elulookirjeldusse laste arvu kirjutamisest numbrit , kuigi ka seda ei peaks tegelikult tegema .","['Tavaliselt ei tee naised elulookirjeldusse laste arvu kirjutamisest numbrit.', 'Ka seda ei peaks tegelikult tegema.']",,L,elt_n80_elt,y,y
189,8879090,14267123,6,vastama,,in,lahendusvariant,lahendusvariandis,Ilmselt me sellele oma detailplaneeringu lahendusvariandis ka vastame .,['Me vastame sellele oma detailplaneeringu lahendusvariandis.'],,L,elt_n80_elt,y,y
163,676751,1076178,14,tahtma,,in,Stockholm,Stockholmis,"Lähipäevil peaks uus Apple iMac siiski kohale jõudma , sest te ju tahate Stockholmis digitaalse videokaameraga salvestatud koduvideo ja fotod arvuti abil monteerida ning mõnele sõbrale vaatamiseks saata .","['Lähipäevil peaks uus Apple iMac kohale jõudma.', 'Te tahate Stockholmis digitaalse videokaameraga salvestatud koduvideo monteerida.', 'Te tahate Stockholmis digitaalse videokaameraga salvestatud fotod monteerida.', 'Te tahate need arvuti abil mõnele sõbrale vaatamiseks saata.']",|L|AL|EL|LS|LT|ALT|ELT|LST|,L,elt_n80_elt,y,y
159,11878378,19007224,5,tüürima,,ad,MM,MMil,Cipollini tüüris oma esimesel MMil meistritiitlile .,['Cipollini tüüris oma esimesel MMil meistritiitlile.'],|E|AE|EL|ET|AET|ELT|,E,elt_n80_elt,y,y
130,18556819,28087743,8,leidma,üles,el,id.ee,id.ee'st,mida muide otsingut kasutamata mitte kuidagi sealt id.ee'st üles ei leia .,"[""Mida otsingut kasutamata mitte kuidagi sealt id.ee'st üles ei leia.""]",,L,elt_n80_elt,y,y
131,13517648,21620049,5,lahkuma,,abl,imedemaa,Imedemaalt,""" Igaühele , kes Imedemaalt lahkub , peaks mõni oma tehtud asi kätte jääma , ” ütles Matvei .","['Igaühele, kes Imedemaalt lahkub, peaks mõni oma tehtud asi kätte jääma.', 'Matvei ütles seda.']",|L|AL|EL|LS|LT|ALT|ELT|LST|,L,elt_n80_elt,y,y
190,13165850,21064784,11,jätkuma,,ill,keskus,keskustesse,"Paisuots ütles , et kuna puudub väljaõppestruktuur , ei jätku keskustesse mehi .","['Paisuots ütles, et puudub väljaõppestruktuur.', 'Paisuots ütles, et keskustesse ei jätku mehi.']",,L,elt_n80_elt,y,y
147,2374958,3805890,41,mööduma,,ad,maantee,maanteel,""" Õnneks on inimesed väga mõistlikud ja ei ole massiliselt tulnud oma autodega , mistõttu jätkub suurhalli ümbruses ka parkimiskohti , "" ütles ETAle Falcki avalike suhete juht Indrek Lindsalu , kelle sõnul möödusid mõlemad kontserdid ilma suuremate liiklusummikutega Paldiski maanteel .","['Õnneks on inimesed väga mõistlikud.', 'Inimesed ei ole massiliselt tulnud oma autodega.', 'Suurhalli ümbruses jätkub parkimiskohti.', 'Falcki avalike suhete juht Indrek Lindsalu ütles seda ETAle.', 'Mõlemad kontserdid möödusid ilma suuremate liiklusummikuteta Paldiski maanteel.']",,L,elt_n80_elt,y,y


In [29]:
same_sent_replaced 

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,simple_sentences_gpt4o,tags,tag,initial_cat,verb_head_same_sent,head_verbi_alluv
118,14457094,22879951,42,käsitlema,,in,õhuveondus,õhuveonduses,"Et tagada lepinguosaliste vahelise veonduse kooskõlaline areng , mis on kohandatud nende kaubandusvajadustele , võivad lepinguosalised pärast käesoleva lepingu jõustumist sõlmida vajaduse korral erilepinguid , mis käsitlevad vastastikuse turulepääsu ja teenuste osutamise tingimusi maantee- , raudtee- ja sisevee- ning vajaduse korral õhuveonduses .","['Lepinguosalised soovivad tagada veonduse kooskõlalise arengu.', 'Veondus peab olema kohandatud lepinguosaliste kaubandusvajadustele.', 'Lepinguosalised võivad sõlmida erilepinguid.', 'Erilepingud käsitlevad vastastikuse turulepääsu ja teenuste osutamise tingimusi.', 'Erilepingud võivad puudutada maantee-, raudtee-, sisevee- ja õhuveondust.', 'Erilepingud sõlmitakse pärast käesoleva lepingu jõustumist.']",,L,elt_n80_elt,y,?


In [30]:
same_sent_kesksona 

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,simple_sentences_gpt4o,tags,tag,initial_cat,verb_head_same_sent,head_verbi_alluv


In [31]:
same_sent_error 

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,simple_sentences_gpt4o,tags,tag,initial_cat,verb_head_same_sent,head_verbi_alluv
105,2021944,3226882,29,juhtima,,el,arvamusküsitlus,arvamusküsitlustest,"Uuringu eestvedaja USA endine välisminister Madeleine Albright nimetas tulemusi jahmatavateks ja raskesti allaneelatavaiks , kuid president George Bush suhtus asjasse rahulikult : "" Ma ei juhi oma valitsust arvamusküsitlustest lähtuvalt .","['Madeleine Albright on uuringu eestvedaja.', 'Madeleine Albright on USA endine välisminister.', 'Madeleine Albright nimetas uuringu tulemusi jahmatavateks.', 'Madeleine Albright nimetas tulemusi raskesti allaneelatavaiks.', 'President George Bush suhtus uuringu tulemustesse rahulikult.', 'President George Bush ütles, et ta ei juhi oma valitsust arvamusküsitlustest lähtuvalt.']",|E|AE|EL|ET|AET|ELT|,L,elt_n80_elt,y,e
150,252869,402877,6,viima,,adit,diagramm,diagrammi,"See suund viiks meid Venni diagrammi , Karnaugh kaardi ja taoliste skeemide poole .","['See suund viiks meid Venni diagrammi poole.', 'See suund viiks meid Karnaugh kaardi poole.', 'See suund viiks meid taoliste skeemide poole.']",,L,elt_n80_elt,y,e
186,12928509,20681112,6,juhtima,,adit,MM-sari,MM-sarja,"Pärast Rootsi rallit juhtis soomlane MM-sarja , praegu , nelja etapi järel , on ta neljandal kohal .","['Pärast Rootsi rallit juhtis soomlane MM-sarja.', 'Praegu, nelja etapi järel, on ta neljandal kohal.']",|E|AE|EL|ET|AET|ELT|,E,elt_n80_elt,y,e


In [29]:
#errors

In [32]:
gpt_errors

,sentence_id,head_id,head_loc,verb,verb_compound,morph_case,lemma,form,sentence,simple_sentences_gpt4o,tags,tag,initial_cat,verb_head_same_sent,head_verbi_alluv
121,3002894,4816565,4,täienema,,ad,avastamine,avastamisel,Kehavälise viljastamise meetodi avastamisel järel on see pidevalt täienenud ning lisandunud on palju muid võimalusi .,"['Kehavälise viljastamise meetod avastati.', 'Pärast avastamist on meetod pidevalt täienenud.', 'Lisandunud on palju muid võimalusi.']",,E,elt_n80_elt,?,?
